# ML Pipeline — Training, Batch Predict & Tests

This notebook runs the full ML pipeline locally and executes the test suite. Run cells top-to-bottom.

**Prerequisite:** Run `02_devcontainer_setup.ipynb` first to confirm the environment is ready.

---

## Pipeline stages
```
data → preprocessing → training → evaluation → artifact
       (src/data.py)  (src/features.py)  (src/train.py)  (src/evaluate.py)
                                                                  ↓
                                                     artifacts/model.pkl
                                                     artifacts/metrics.json
```

In [ ]:
import sys
import os

# Ensure project root is on the path
ROOT = "/workspaces/marketing-model-mlops-azure"
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print(f"Working directory: {os.getcwd()}")

## 1. Train the Model

Runs the full pipeline: load → feature engineering → train → evaluate → save artifact.

**Outputs:**
- `artifacts/model.pkl` — trained sklearn Pipeline
- `artifacts/metrics.json` — ROC-AUC, F1, and additional metrics

In [ ]:
%%bash
cd /workspaces/marketing-model-mlops-azure
python main.py train

In [ ]:
# Verify artifacts were created
import json

model_path = os.path.join(ROOT, "artifacts/model.pkl")
metrics_path = os.path.join(ROOT, "artifacts/metrics.json")

print(f"model.pkl exists:   {os.path.exists(model_path)}")
print(f"metrics.json exists:{os.path.exists(metrics_path)}")

if os.path.exists(metrics_path):
    print()
    print("=== Metrics ===")
    with open(metrics_path) as f:
        metrics = json.load(f)
    for k, v in metrics.items():
        print(f"  {k}: {v}")

## 2. Inspect the Trained Model

Load and inspect the artifact to confirm it is a valid sklearn pipeline.

In [ ]:
import joblib

model = joblib.load(os.path.join(ROOT, "artifacts/model.pkl"))
print(f"Model type: {type(model)}")
print()
print("Pipeline steps:")
for name, step in model.steps:
    print(f"  {name}: {type(step).__name__}")

## 3. Batch Predict

Run inference on the full dataset CSV and save results to `data/results/predictions.csv`.

In [ ]:
%%bash
cd /workspaces/marketing-model-mlops-azure

# Print predictions to stdout (first 5 lines)
python main.py predict \
  --input data/raw/bank_marketing_data.csv \
  --output data/results/predictions.csv

echo ""
echo "Output file:"
head -5 data/results/predictions.csv

In [ ]:
# Inspect predictions in Python
import pandas as pd

preds = pd.read_csv(os.path.join(ROOT, "data/results/predictions.csv"))
print(f"Prediction rows: {len(preds)}")
print(f"Columns: {list(preds.columns)}")
print()
print(preds.head())

## 4. Run All Tests

Runs the full pytest suite. A trained `artifacts/model.pkl` is required for API tests — run **Section 1** first.

| Test module | What it covers |
|---|---|
| `test_config.py` | Config loading, required keys, missing file handling |
| `test_data.py` | DataFrame loading, row counts, column validation |
| `test_features.py` | Cleaning transforms, split sizes, preprocessor construction |
| `test_api.py` | `/health`, `/predict`, field validation, label consistency |

In [ ]:
%%bash
cd /workspaces/marketing-model-mlops-azure
python -m pytest tests/ -v --tb=short

## 5. Run Individual Test Modules

Run a specific test module to isolate failures. Uncomment the module you want to run.

In [ ]:
%%bash
cd /workspaces/marketing-model-mlops-azure

# Uncomment the test module to run:
python -m pytest tests/test_config.py -v
# python -m pytest tests/test_data.py -v
# python -m pytest tests/test_features.py -v
# python -m pytest tests/test_api.py -v

## 6. Direct Inference (Python)

Call the model directly without starting the API server — useful for quick debugging.

In [ ]:
import joblib
import pandas as pd
from src.config import load_config
from src.features import clean_data

config = load_config(os.path.join(ROOT, "config.yaml"))
model = joblib.load(os.path.join(ROOT, "artifacts/model.pkl"))

# Raw input — same 16 columns the API /predict endpoint accepts
sample = pd.DataFrame([{
    "age": 35,
    "job": "management",
    "marital": "married",
    "education": "tertiary",
    "default": "no",
    "balance": 1500.0,
    "housing": "yes",
    "loan": "no",
    "contact": "cellular",
    "day": 15,
    "month": "may",
    "duration": 250.0,
    "campaign": 1,
    "pdays": -1,
    "previous": 0,
    "poutcome": "unknown",
}])

# Apply the same feature engineering the API uses before calling the Pipeline
# clean_data() derives 'contacted_before' from pdays and applies log transforms
sample_clean = clean_data(sample, config)

pred = model.predict(sample_clean)[0]
prob = model.predict_proba(sample_clean)[0][1]
label = "yes" if pred == 1 else "no"

print(f"prediction:  {pred}")
print(f"probability: {prob:.4f}")
print(f"label:       {label}")

---

## Summary

| Step | Command | Expected result |
|---|---|---|
| Train | `python main.py train` | `artifacts/model.pkl` created |
| Metrics | `artifacts/metrics.json` | ROC-AUC, F1 scores printed |
| Batch predict | `python main.py predict --input ... --output ...` | CSV with predictions |
| All tests | `pytest tests/ -v` | All tests pass |

Once tests pass and the artifact is confirmed, open **`04_docker_testing.ipynb`** to build and test the Docker image.